# Brain Tumor Classification
**Estudio comparativo de 5 arquitecturas Deep Learning**

---
### Instrucciones:
1. En la **Celda 3** cambia `MODELO` y `FASE`
2. Ejecuta todo en orden (no te saltes pasos)
3. Al finalizar, los resultados se guardan automáticamente en Drive

---
## 0. Montar Google Drive

Permite el acceso cuando Google lo pida.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## 1. Configuración del experimento

**Modelos disponibles:**
- `"resnet50"` — CNN clásica (Residual Learning)
- `"efficientnet_b0"` — CNN optimizada (Compound Scaling)
- `"vit"` — Vision Transformer (Self-Attention global)
- `"swin"` — Swin Transformer (Ventanas desplazables)
- `"coatnet"` — Híbrido CNN+Transformer

**Fases:**
- `1` — **Base**: sin augmentation, sin early stopping, sin scheduler
- `2` — **Optimizado**: con augmentation, early stopping, scheduler, mixed precision

In [ ]:
# ========================================
# TU SOLO CAMBIA ESTO:
# ========================================
MODELO = "resnet50"   # <-- modelo a entrenar
FASE = 2              # <-- 1 = Base, 2 = Optimizado

# Rutas en tu Google Drive (ajústalas si es necesario)
RUTA_DRIVE = "/content/drive/MyDrive/BrainTumor"
RUTA_DATASET = "/content/drive/MyDrive/BrainTumor_Dataset"
# ========================================

print(f"Modelo: {MODELO}")
print(f"Fase:   {'2 - Optimizado' if FASE == 2 else '1 - Base'}")
print(f"Drive:  {RUTA_DRIVE}")
print(f"Dataset: {RUTA_DATASET}")

---
## 2. Copiar proyecto y dataset a Colab

Esto acelera el acceso a los archivos.

In [ ]:
import os, shutil

# Copiar proyecto
if os.path.exists("/content/BrainTumor"):
    shutil.rmtree("/content/BrainTumor")
shutil.copytree(RUTA_DRIVE, "/content/BrainTumor")
os.chdir("/content/BrainTumor")
print("Proyecto copiado")

# Copiar dataset dentro del proyecto
dst = "/content/BrainTumor/dataset/BrainTumor_Dataset"
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(RUTA_DATASET, dst)
print("Dataset copiado")

print(f"\nContenido de dataset/: {os.listdir('dataset')}")

---
## 3. Instalar dependencias

Colab ya trae PyTorch. Instalamos lo que falta.

In [ ]:
!pip install -q matplotlib scikit-learn tqdm

# Si el modelo es coatnet, instala timm
if MODELO == "coatnet":
    !pip install -q timm

print("Dependencias listas")

---
## 4. Verificar GPU

Debe mostrar **`True`** y el nombre de la GPU (ej. Tesla T4).

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No hay GPU. Entrenamiento lento.")
    print("  -> Ve a: Entorno de ejecucion -> Cambiar tipo -> T4 GPU")

---
## 5. Análisis Exploratorio de Datos (EDA)

Antes de entrenar, entendemos el dataset.

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

dataset_root = Path("dataset/BrainTumor_Dataset")

# Distribución por split y clase
print("=" * 50)
print("DISTRIBUCIÓN DEL DATASET")
print("=" * 50)
total = 0
for split in ["train", "val", "test"]:
    split_path = dataset_root / split
    classes = sorted(split_path.iterdir())
    counts = [len(list(c.iterdir())) for c in classes]
    split_total = sum(counts)
    for c, cnt in zip(classes, counts):
        pct = cnt / split_total * 100
        print(f"  {split:8s} / {c.name:6s}: {cnt:5d}  ({pct:5.1f}%)")
    total += split_total
print(f"  {'TOTAL':8s}: {total:5d}")
# Desbalance por split
print()
for split in ["train", "val", "test"]:
    split_path = dataset_root / split
    class_counts = {}
    for class_dir in sorted(split_path.iterdir()):
        class_counts[class_dir.name] = len(list(class_dir.iterdir()))
    max_c = max(class_counts.values())
    min_c = min(class_counts.values())
    ratio = max_c / min_c
    print(f"  {split:8s}: ratio yes/no = {ratio:.2f}x  ({(max_c - min_c):d} img de diferencia)")
print()

# Ejemplos de imágenes
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for i, clase in enumerate(["no", "yes"]):
    class_path = dataset_root / "train" / clase
    img_path = list(class_path.iterdir())[0]
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(f"Clase: {clase}\nTamaño: {img.size}")
    axes[i].axis("off")
plt.tight_layout()
plt.savefig("results/eda_samples.png", bbox_inches="tight")
plt.show()
print("\nImágenes de ejemplo guardadas en results/eda_samples.png")

---
## 6. Limpieza del dataset

Verificamos que no haya imágenes corruptas.

In [ ]:
!python src/clean_dataset.py

In [ ]:
# Verificar distribución después de la limpieza
from pathlib import Path
dataset_root = Path("dataset/BrainTumor_Dataset")
print("Distribución final tras limpieza:")
print("-" * 55)
total_final = 0
for split in ["train", "val", "test"]:
    split_path = dataset_root / split
    classes = sorted(split_path.iterdir())
    counts = [len(list(c.iterdir())) for c in classes]
    split_total = sum(counts)
    for c, cnt in zip(classes, counts):
        print(f"  {split:8s} / {c.name:6s}: {cnt:5d}")
    print(f"  {'':8s}  Subtotal: {split_total:5d}")
    total_final += split_total
print(f"  {'TOTAL':8s}: {total_final:5d} imágenes")


---
## 7. Entrenamiento

### Fase 1 — Base (~15-30 min)
- Sin data augmentation
- Sin early stopping
- Sin scheduler
- Comparación justa entre arquitecturas

### Fase 2 — Optimizado (~30-60 min)
- Con data augmentation (solo train)
- Early stopping (paciencia = 7)
- Learning Rate Scheduler (ReduceLROnPlateau)
- Mixed Precision (AMP) para GPU

---
### Prueba rápida (2 épocas)

In [ ]:
from pathlib import Path
phase_tag = "optimized" if FASE == 2 else "base"
modelo_path = Path(f"results/{MODELO}/{phase_tag}/best_model.pth")
if modelo_path.exists():
    print(f"Saltando prueba rapida - ya existe {modelo_path}")
else:
    print("Ejecutando prueba rápida de 2 épocas...")
    !python src/train.py --model $MODELO --phase $FASE --epochs 2
    print("[OK] Prueba exitosa" if modelo_path.exists() else "[ERROR] fallo")

---
### Entrenamiento completo

Ejecuta solo si la prueba rápida funcionó.

In [ ]:
!python src/train.py --model $MODELO --phase $FASE
print("\n[OK] Entrenamiento completado")

---
## 8. Evaluación

Métricas de rendimiento y computacionales:
- Accuracy, Precision, Recall, F1, ROC-AUC
- Sensitivity, Specificity, Balanced Accuracy, MCC
- Parámetros totales, tiempo de inferencia

In [ ]:
!python src/evaluate.py --model $MODELO --phase $FASE
print("\n[OK] Evaluacion completada")

---
## 9. Resultados

Métricas y gráficas del modelo entrenado.

In [ ]:
import json

phase_tag = "optimized" if FASE == 2 else "base"
ruta = Path(f"results/{MODELO}/{phase_tag}")

if (ruta / "metrics.json").exists():
    with open(ruta / "metrics.json") as f:
        metrics = json.load(f)
    print(f"Métricas de {MODELO} (Fase {FASE}):")
    print("-" * 40)
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k:30s}: {v:.4f}")
        else:
            print(f"  {k:30s}: {v}")
else:
    print("Todavía no hay métricas. Ejecuta Entrenamiento + Evaluación.")

# Cargar historial y graficar curvas de aprendizaje
if (ruta / "history.json").exists():
    with open(ruta / "history.json") as f:
        history = json.load(f)
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(epochs, history["train_loss"], label="Train Loss")
    ax1.plot(epochs, history["val_loss"], label="Val Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(alpha=0.3)
    if "train_acc" in history and len(history["val_metrics"]) > 0:
        val_acc = [m["accuracy"] for m in history["val_metrics"]]
        ax2.plot(epochs, history["train_acc"], label="Train Acc")
        ax2.plot(epochs, val_acc, label="Val Acc")
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("Accuracy")
        ax2.legend()
        ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(ruta / "learning_curves.png", bbox_inches="tight")
    plt.show()

# Mostrar gráficas
from IPython.display import Image, display
for img in ["confusion_matrix.png", "roc_curve.png"]:
    path = ruta / img
    if path.exists():
        display(Image(filename=str(path)))

---
## 10. Guardar resultados en Drive

Copia los resultados para no perderlos al cerrar Colab.

In [ ]:
phase_tag = "optimized" if FASE == 2 else "base"
src = f"/content/BrainTumor/results/{MODELO}/{phase_tag}"
dst = f"{RUTA_DRIVE}/results/{MODELO}/{phase_tag}"

os.makedirs(dst, exist_ok=True)
shutil.copytree(src, dst, dirs_exist_ok=True)
print(f"Resultados guardados en: {dst}")
print("Ya puedes cerrar Colab.")